# OCF / Ghost Density — Scorer Update 10/2

OCF/Ghost Density Scorer
This measures a specific compositional pattern where generative models maintain coherence under offset constraints:

- Subject displaced approx 1/6 frame width from anchor (edge/center)
- Void ratio 2/3 (subject occupies 1/3 of frame)
- Surface roughness in controlled band (~0.28–0.40)
- Optional structural anchor (vertical edge or horizon line)
**This tool identifies a specific geometric pattern, not compositional quality.** Images outside this pattern may be well-composed using different strategies.

Most diffusion models collapse to centered, symmetrical compositions. This basin represents a reproducible alternative attractor — configurations that remain stable despite asymmetry. Not "better composition" or "aesthetic judgement," just a measurable geometry that avoids model collapse.
Three profile variants:

GD-Edge: Figure offset from vertical wall/seam (portraits, figure to wall, figures interiors)
GD-Horizon: Subject in field with horizon anchor (landscape, sea/sky, fields w/subject, still life)
Field-Void: Unanchored subject in open field (single object in space, voids)

What it doesn't do:

- Judge aesthetic quality
- Handle centered compositions
- Analyze distributed/mosaic layouts
- Work with abstract or pattern-field images
- Evaluate compositions outside this specific geometry

Core metrics:

- Δx: Horizontal displacement (from seam or center)
- r_v: Void ratio (background/total area)
- σ_r (pr): Edge density after perceptual normalization

Output:
- Pass/fail for "Artist Basin" (tight), "Engine Window" (production-tolerant) and (mid) and "Orbit" (close proximity) bands
- Diagnostic deltas showing distance from target ranges.

Use case: Filter/score generative outputs for this specific compositional pattern, or provide feedback during generation steering.

Most models collapse to *safe center*. OCF reframes that as **geography or basin**: there are reproducible *attractors* where off-center images remain coherent. By measuring Δx, rᵥ, ρᵣ and applying small, engine-aware nudges (plus a one-click crop), you can hit these three observable basins **reliably**—and explain *why* a result passed or failed.

## Calibration Notes

### Area Correction Factor (0.55)

The `area = area * 0.55` multiplier in `compute_metrics()` compensates for systematic over-estimation by auto-masking on gradient-heavy images.

**Empirical basis:** Auto-masking (Otsu + k-means LAB fallback) groups soft tonal transitions and atmospheric gradients as "subject" rather than "void." On images with continuous tonal fields (common in Sora outputs, Morandi-style compositions), measured subject area averages 1.8× true area.

**Derivation:** Calibrated against visual inspection of 200+ exemplar images where ground-truth void ratio should be ~70% (subject ~30%), but auto-masks measured ~50% subject. Correction factor: 0.30 / 0.55 ≈ 0.55 (target subject area / measured subject area). May need to be adjusted based on image/engine.

**Validity domain:** This constant is specific to:
- Off-center compositional patterns with ambiguous figure/ground
- Low-contrast, gradient-rich backgrounds
- Images where k-means chromatic fallback is triggered

For high-contrast images with sharp subject/void boundaries (where Otsu succeeds), the correction has minimal impact as measured area is already accurate.

**Alternative approach:** Manually mask all reference images. This constant exists because auto-masking is a convenience tool for production use, not ground truth. If measurements disagree with visual assessment, manual masks override auto-masking.

**Calibration note:** This factor was tuned for AI-generated images (particularly Sora/ChatGPT outputs with gradient backgrounds, but validated across Gemini, Meta, MidJourney, Stable Diffusion, FireFly, Canva). Photographs or high-contrast renders may require different correction factors. Test on representative samples and adjust if systematic bias persists.

In [ ]:
# OCF / Ghost Density — Administrator canonical primitives

import io, os, math, zipfile, numpy as np, cv2, pandas as pd
from PIL import Image, UnidentifiedImageError

# ====== CONFIG ======
PR_MODE = "legacy"          # "legacy" | "normalized" | "bg_only"
DEFAULT_MASK_POLARITY = "auto"  # "Lighter" | "darker" | "lighter"

# ---- EDGE PREP (adaptive) ----
EDGE_RESIZE_MAX = 864      # downscale longest side before edge-finding (keeps edges “thick”)
EDGE_USE_CLAHE  = True      # boost local contrast for flat walls/voids Enable for low-contrast walls/voids (flat backgrounds)
                             # Increases edge detection sensitivity but may find spurious lines
EDGE_PERCENTILE = 90        # Top N% of gradients = edges (adaptive per image)

# Document-faithful bands
ART_EDGE   = {"dx":(0.15,0.18), "rv":(0.62,0.68), "pr":(0.28,0.40)}
ENG_EDGE   = {"dx":(0.12,0.20), "rv":(0.62,0.68), "pr":(0.28,0.40)}
HORIZON_RV = (0.70,0.85); HORIZON_PR=(0.18,0.30)
FIELD_RV   = (0.74,0.90); FIELD_PR  =(0.18,0.32)

# ====== HELPERS ======
def clip01(x):
    try: return float(min(1.0, max(0.0, x)))
    except: return float('nan')

def auto_mask_fast(bgr, polarity=DEFAULT_MASK_POLARITY):
    g   = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    g   = cv2.GaussianBlur(g, (5,5), 1.2)
    cla = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(g)
    _, bw = cv2.threshold(cla, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    if polarity == 'auto':
        m_fg = float(g[bw>0].mean())  if (bw>0).any()  else 128.0
        m_bg = float(g[bw==0].mean()) if (bw==0).any() else 128.0
        if m_fg > m_bg + 3.0:
            bw = 255 - bw
    elif polarity == 'lighter':
        bw = 255 - bw
    cnts,_ = cv2.findContours(bw, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    m = np.zeros_like(bw)
    if cnts:
        c = max(cnts, key=cv2.contourArea)
        cv2.drawContours(m, [c], -1, 255, -1)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((3,3),np.uint8), 1)
    return m

def auto_mask_chromatic(bgr, polarity="auto"):
    """
    Subject/void segmentation using color contrast, not just luminance.
    Handles Morandi-style subtle value gradients.
    """
    # Convert to LAB - separates luminance from chrominance
    lab = cv2.cvtColor(bgr, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)

    # Cluster in color space (k=2 for subject/void)
    pixels = lab.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 10, 1.0)
    _, labels, centers = cv2.kmeans(pixels, 2, None, criteria, 3, cv2.KMEANS_PP_CENTERS)

    # Reshape to image
    mask = labels.reshape(bgr.shape[:2])

    # Determine which cluster is subject (assume more compact/less edge-touching)
    mask0 = (mask == 0).astype(np.uint8)
    mask1 = (mask == 1).astype(np.uint8)

    # Subject is typically not at edges
    edge_margin = max(bgr.shape[:2]) // 20
    border_mask = np.zeros_like(mask0)
    border_mask[:edge_margin, :] = 1
    border_mask[-edge_margin:, :] = 1
    border_mask[:, :edge_margin] = 1
    border_mask[:, -edge_margin:] = 1

    border_0 = (mask0 * border_mask).sum()
    border_1 = (mask1 * border_mask).sum()

    subject_mask = mask1 if border_0 > border_1 else mask0

    # Clean up
    subject_mask = cv2.morphologyEx(subject_mask * 255, cv2.MORPH_CLOSE,
                                     np.ones((5,5), np.uint8))

    return subject_mask


def auto_mask_smart(bgr, polarity=DEFAULT_MASK_POLARITY):
    """Try Otsu first, fall back to k-means if quality is poor."""

    # Try Otsu (fast)
    mask_otsu = auto_mask_fast(bgr, polarity)
    area, convexity = mask_quality(mask_otsu)

    # Quality checks
    otsu_ok = (
        0.10 < area < 0.60 and  # Subject is 10-60% of frame
        convexity > 0.75         # Subject is reasonably compact
    )

    if otsu_ok:
        return mask_otsu, "otsu"

    # Fall back to k-means LAB (slower but handles gradients/chrominance)
    return auto_mask_chromatic(bgr, polarity), "kmeans"

def ensure_mask2d(maybe_mask, bgr, polarity=DEFAULT_MASK_POLARITY):
    if maybe_mask is None:
        mask, method = auto_mask_smart(bgr, polarity=polarity)
        return mask
    m = np.array(maybe_mask)
    if m.ndim == 3: m = cv2.cvtColor(m, cv2.COLOR_BGR2GRAY)
    return ((m>127).astype(np.uint8))*255

def mask_quality(mask_uint8):
    if mask_uint8 is None: return 0.0, 0.0
    h,w = mask_uint8.shape[:2]
    area = (mask_uint8>0).sum() / float(h*w)
    cnts,_ = cv2.findContours(mask_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts: return area, 0.0
    c=max(cnts, key=cv2.contourArea); hull=cv2.convexHull(c)
    a=max(cv2.contourArea(c),1.0); ah=max(cv2.contourArea(hull),1.0)
    return area, a/ah

# ====== LINES / SEAMS ======
def detect_lines(bgr, vertical=True, use_clahe=False, span_frac=0.45):
    img = bgr.copy()
    if use_clahe:
        g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(g)
        img = cv2.cvtColor(g, cv2.COLOR_GRAY2BGR)
    h,w = img.shape[:2]
    edges = cv2.Canny(cv2.GaussianBlur(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY),(3,3),0.8), 60, 180)
    lines = cv2.HoughLinesP(edges, 1, np.pi/180, threshold=60,
                            minLineLength=int(span_frac*(h if not vertical else w)),
                            maxLineGap=12)
    ret=[]
    if lines is not None:
        for l in lines[:,0]:
            x1,y1,x2,y2 = map(int,l)
            dx,dy=abs(x2-x1),abs(y2-y1)
            if vertical  and dy>=dx*2: ret.append((x1,y1,x2,y2))
            if not vertical and dx>=dy*2: ret.append((x1,y1,x2,y2))
    return ret

def pick_nearest_vertical(vlines, cx):
    if not vlines: return None
    return min(vlines, key=lambda L: abs(0.5*(L[0]+L[2]) - cx))

# ====== METRICS (Δx, r_v, ρ_r) with PR_MODE ======
def compute_metrics(bgr, mask, profile, anchors, use_clahe=EDGE_USE_CLAHE):
    # --- Process at full resolution with perceptual normalization ---
    bgr_s = bgr.copy()
    H, W = bgr_s.shape[:2]
    scale = 1.0  # No downsampling - perceptual units handle scale

    # --- mask & centroid (compute on resized) ---
    m_s  = ensure_mask2d(mask, bgr_s)
    ys, xs = np.nonzero(m_s)
    cx = float(xs.mean()) if xs.size else W/2.0
    cy = float(ys.mean()) if ys.size else H/2.0

    # r_v (void ratio) on resized
    area = (m_s>0).sum()/float(H*W)
    area = area * 0.55
    rv   = clip01(1.0 - area)

    # --- unified robust edge detection ---
    g = cv2.cvtColor(bgr_s, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (3,3), 1.2)
    if use_clahe:
        g = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8)).apply(g)

    # Structure tensor approach - better for compositional edges
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)
    gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    magnitude = np.sqrt(gx**2 + gy**2)

    # Adaptive threshold: top 10% of gradients
    # (compositional edges are structural, not everywhere)
    threshold = np.percentile(magnitude, EDGE_PERCENTILE)
    edges = (magnitude > threshold).astype(np.uint8) * 255

    # Connect interrupted seams
    edges = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, np.ones((3,3), np.uint8))

    # --- ρᵣ modes ---
    # --- σ_r modes (scale-invariant) ---
    # Normalize kernel by viewing distance / image diagonal
    diag = np.sqrt(H**2 + W**2)

    # Perceptual unit: at comfortable viewing distance (~1.5× diagonal),
    # viewer resolves detail at ~1/100th of diagonal as minimum feature
    perceptual_unit = diag / 100.0

    if PR_MODE == "legacy":
        # Dilate kernel sized to perceptual units (not arbitrary % of dimensions)
        k = max(3, int(round(perceptual_unit * 0.5)))  # Half a perceptual unit
        if k % 2 == 0: k += 1
        e01 = (edges > 0).astype(np.uint8)
        e01 = cv2.dilate(e01, np.ones((k, k), np.uint8), 1)
        pr  = clip01(e01.mean().item())

    elif PR_MODE == "bg_only":
        # Same perceptual normalization for bg_only mode
        k = max(3, int(round(perceptual_unit * 0.5)))
        if k % 2 == 0: k += 1
        e01 = (edges > 0).astype(np.uint8)
        e01 = cv2.dilate(e01, np.ones((k, k), np.uint8), 1)

        bg     = 1.0 - (m_s.astype(np.float32)/255.0)
        bg_area= bg.mean() + 1e-6
        pr     = clip01(((e01.astype(np.float32) * bg).mean()) / bg_area)

    else:  # "normalized"
        # For normalized mode, still apply perceptual dilate
        k = max(3, int(round(perceptual_unit * 0.5)))
        if k % 2 == 0: k += 1
        e01 = (edges > 0).astype(np.uint8)
        e01 = cv2.dilate(e01, np.ones((k, k), np.uint8), 1)
        pr  = clip01(e01.mean().item())

    CAL_SLOPE, CAL_INTERCEPT = 1.013, 0.0
    pr = float(np.clip(CAL_SLOPE * pr + CAL_INTERCEPT, 0.0, 1.0))

    # --- Δx: seam (if Edge & found) else center ---
    if profile == "GD-Edge" and anchors and anchors.get("vline") is not None:
        x1,y1,x2,y2 = anchors["vline"]
        seam_x = 0.5*(x1 + x2)
        dx = abs(cx - seam_x)/W
    else:
        dx = abs(cx - (W/2.0))/W

    return float(dx), float(rv), float(pr), (cx, cy)

# ====== ENVELOPES & SCORING ======
def in_band(v, band): return band[0] <= v <= band[1]
def dist_to_band(v, band):
    lo,hi = band
    if v<lo: return (lo-v)/(hi-lo), 0.0
    if v>hi: return 0.0, (v-hi)/(hi-lo)
    return 0.0, 0.0

def score_image(dx, rv, pr, profile, anchors_ok=True):
    if profile=="GD-Edge":
        bandsA, bandsE = ART_EDGE, ENG_EDGE; req_anchor=True
    elif profile=="GD-Horizon":
        bandsA = {"dx":(0.15,0.20),"rv":HORIZON_RV,"pr":HORIZON_PR}
        bandsE = {"dx":(0.12,0.22),"rv":HORIZON_RV,"pr":(0.18,0.32)}; req_anchor=False
    else: # Field-Void
        bandsA = {"dx":(0.00,1.00),"rv":FIELD_RV,"pr":FIELD_PR}
        bandsE = {"dx":(0.00,1.00),"rv":(0.74,0.88),"pr":(0.20,0.34)}; req_anchor=False

    artist = (in_band(dx,bandsA["dx"]) and in_band(rv,bandsA["rv"]) and in_band(pr,bandsA["pr"])
              and (anchors_ok or not req_anchor))
    deploy = (in_band(dx,bandsE["dx"]) and in_band(rv,bandsE["rv"]) and in_band(pr,bandsE["pr"])
              and (anchors_ok or not req_anchor))

    dx_out = dist_to_band(dx, bandsE["dx"])
    rv_out = dist_to_band(rv, bandsE["rv"])
    pr_out = dist_to_band(pr, bandsE["pr"])
    return artist, deploy, dx_out, rv_out, pr_out

# ====== AUTO PROFILE GUESS + OVERLAY ======
def guess_profile(bgr, mask, use_clahe=False, span_frac=0.45):
    """
    Fuzzy profile classification with overlap zones.
    Returns best guess + probability distribution for ambiguous cases.
    """
    m = ensure_mask2d(mask, bgr)
    area, _ = mask_quality(m)
    rv_est = clip01(1.0 - area)

    vlines = detect_lines(bgr, vertical=True, use_clahe=use_clahe, span_frac=span_frac)
    hlines = detect_lines(bgr, vertical=False, use_clahe=use_clahe, span_frac=span_frac)

    # Gaussian membership functions for smooth boundaries
    def gauss_membership(x, center, width):
        return np.exp(-0.5 * ((x - center) / width) ** 2)

    # GD-Edge: peaks at rv=0.65, requires vertical line
    edge_rv_score = gauss_membership(rv_est, 0.65, 0.08)
    best_len_px = 0.0
    W = bgr.shape[1]
    for ln in (vlines or []):
        try:
            # support tuple/list or dict line formats
            if isinstance(ln, (list, tuple)) and len(ln) >= 4:
                x1, y1, x2, y2 = ln[:4]
            else:
                x1 = int(ln["x1"]); y1 = int(ln["y1"]); x2 = int(ln["x2"]); y2 = int(ln["y2"])
            length = float(np.hypot(x2 - x1, y2 - y1))
            if length > best_len_px:
                best_len_px = length
        except Exception:
            pass

    # require BOTH: (a) span gate and (b) absolute strength gate
    meets_span      = best_len_px >= (span_frac * W)   # UI “Span” is the min line length
    meets_strength  = best_len_px >= (0.60 * W)        # tune 0.60–0.70 if needed

    edge_vline_score = 1.0 if (meets_span and meets_strength) else 0.0
    edge_score = edge_rv_score * edge_vline_score

    # GD-Horizon: peaks at rv=0.77, benefits from horizontal line
    horiz_rv_score = gauss_membership(rv_est, 0.77, 0.10)
    horiz_hline_score = 1.0 if len(hlines) > 0 else 0.3
    horiz_score = horiz_rv_score * horiz_hline_score

    # Field-Void: peaks at rv=0.82, no anchor requirement
    field_rv_score = gauss_membership(rv_est, 0.82, 0.10)
    field_score = field_rv_score

    # Normalize to probabilities
    total = edge_score + horiz_score + field_score + 1e-6
    probs = {
        "GD-Edge": edge_score / total,
        "GD-Horizon": horiz_score / total,
        "Field-Void": field_score / total,
    }

    # Select highest probability profile
    prof = max(probs.items(), key=lambda x: x[1])[0]

    info = {
        "edge_conf": probs["GD-Edge"],
        "horiz_conf": probs["GD-Horizon"],
        "field_conf": probs["Field-Void"],
        "probabilities": probs,  # Full distribution
        "is_ambiguous": max(probs.values()) < 0.5,  # Flag close calls
    }

    return prof, info

def draw_overlay(bgr, mask, profile, anchors, dx, rv, pr, cxcy, info=None):
    H,W = bgr.shape[:2]; out=bgr.copy()
    m = ensure_mask2d(mask, bgr)
    if m is not None:
        out = cv2.addWeighted(out,1.0, cv2.merge([(m//6)]*3), 0.35, 0)
    cx,cy = cxcy; cv2.circle(out,(int(cx),int(cy)),4,(0,0,255),-1)
    if profile=="GD-Edge" and anchors and anchors.get("vline") is not None:
        x1,y1,x2,y2 = anchors["vline"]; cv2.line(out,(x1,y1),(x2,y2),(255,0,0),2)
        lx=int(0.5*(x1+x2)); cv2.arrowedLine(out,(int(cx),int(cy)),(lx,int(cy)),(255,255,0),2,tipLength=0.03)
    else:
        cv2.arrowedLine(out,(int(cx),int(cy)),(W//2,int(cy)),(255,255,0),2,tipLength=0.03)
    state = info.get("state", "Unknown") if info else "Unknown"
    txt=f"Δx={dx:.3f}  rᵥ={rv:.3f}  σᵣ={pr:.3f}  {profile}  [{state}]"
    bg_color = get_state_color(state) if 'get_state_color' in dir() else (0,0,0)
    cv2.rectangle(out,(10,10),(10+len(txt)*9,40), bg_color, -1)
    cv2.putText(out,txt,(16,34),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,255),1,cv2.LINE_AA)
    return out

print("Primitives loaded. PR_MODE =", PR_MODE)

Primitives loaded. PR_MODE = legacy


In [ ]:
ORBIT_TOLERANCE = 0.30  # 30% extension beyond Engine Window bands

def classify_state(dx, rv, pr, profile, anchors_ok, artist_pass, deploy_pass, dx_out, rv_out, pr_out):
    """
    Classifies image into one of five states based on basin proximity.

    Returns:
        state (str): "Artist Basin" | "Engine Window" | "Orbit Zone" | "Collapse" | "Dissolution"
        metrics_in_range (int): How many of the 3 metrics are inside Engine Window (0-3)
        closest_metric (str): Which metric needs smallest adjustment to enter Engine Window
    """

    # Quick wins
    if artist_pass:
        return "Artist Basin", 3, None
    if deploy_pass:
        return "Engine Window", 3, None

    # Check for collapse (too centered)
    if dx < 0.05:
        return "Collapse", 0, "dx"

    # Check for dissolution (void lost or surfaces overwhelmed)
    if rv < 0.50 or pr > 0.60:
        return "Dissolution", 0, ("rv" if rv < 0.50 else "pr")

    # Count metrics in Engine Window range
    if profile == "GD-Edge":
        bands = ENG_EDGE
    elif profile == "GD-Horizon":
        bands = {"dx":(0.12,0.22), "rv":HORIZON_RV, "pr":(0.18,0.32)}
    else:  # Field-Void
        bands = {"dx":(0.00,1.00), "rv":(0.74,0.88), "pr":(0.20,0.34)}

    dx_in = in_band(dx, bands["dx"])
    rv_in = in_band(rv, bands["rv"])
    pr_in = in_band(pr, bands["pr"])

    metrics_in = sum([dx_in, rv_in, pr_in])

    # Orbit Zone: 2/3 metrics in range, OR all within tolerance extension
    if metrics_in >= 2:
        # Find which metric is out
        if not dx_in:
            closest = "dx"
        elif not rv_in:
            closest = "rv"
        else:
            closest = "pr"
        return "Orbit Zone", metrics_in, closest

    # Check if within tolerance extension of bands
    dx_width = bands["dx"][1] - bands["dx"][0]
    rv_width = bands["rv"][1] - bands["rv"][0]
    pr_width = bands["pr"][1] - bands["pr"][0]

    dx_extended = (bands["dx"][0] - ORBIT_TOLERANCE * dx_width,
                   bands["dx"][1] + ORBIT_TOLERANCE * dx_width)
    rv_extended = (bands["rv"][0] - ORBIT_TOLERANCE * rv_width,
                   bands["rv"][1] + ORBIT_TOLERANCE * rv_width)
    pr_extended = (bands["pr"][0] - ORBIT_TOLERANCE * pr_width,
                   bands["pr"][1] + ORBIT_TOLERANCE * pr_width)

    if (in_band(dx, dx_extended) and
        in_band(rv, rv_extended) and
        in_band(pr, pr_extended)):
        # Find closest metric to getting in range
        distances = {
            "dx": min(abs(dx - bands["dx"][0]), abs(dx - bands["dx"][1])),
            "rv": min(abs(rv - bands["rv"][0]), abs(rv - bands["rv"][1])),
            "pr": min(abs(pr - bands["pr"][0]), abs(pr - bands["pr"][1]))
        }
        closest = min(distances.items(), key=lambda x: x[1])[0]
        return "Orbit Zone", metrics_in, closest

    # If GD-Edge and no anchor detected
    if profile == "GD-Edge" and not anchors_ok:
        return "Collapse", metrics_in, "anchor"

    # Otherwise unclassified (outside orbit tolerance)
    distances = {
        "dx": min(abs(dx - bands["dx"][0]), abs(dx - bands["dx"][1])),
        "rv": min(abs(rv - bands["rv"][0]), abs(rv - bands["rv"][1])),
        "pr": min(abs(pr - bands["pr"][0]), abs(pr - bands["pr"][1]))
    }
    closest = min(distances.items(), key=lambda x: x[1])[0]
    return "Unclassified", metrics_in, closest

def get_state_color(state):
    """Returns BGR color tuple for overlay background based on state."""
    colors = {
        "Artist Basin": (0, 200, 0),      # Green
        "Engine Window": (200, 100, 0),   # Blue
        "Orbit Zone": (0, 200, 255),      # Yellow
        "Collapse": (0, 0, 200),          # Red
        "Dissolution": (50, 50, 50),      # Dark gray
        "Unclassified": (100, 100, 100)   # Gray
    }
    return colors.get(state, (100, 100, 100))

print("Orbit Zone classification loaded. ORBIT_TOLERANCE =", ORBIT_TOLERANCE)

In [ ]:
# ZIP helpers for batch mode
_IMG_EXTS  = (".png",".jpg",".jpeg",".webp",".bmp",".tif",".tiff")
_MASK_EXTS = (".png",".jpg",".jpeg",".bmp",".tif",".tiff")

def _norm_key(fn):
    base = os.path.basename(fn); return os.path.splitext(base)[0].lower()

def _skip(info):
    fn=info.filename; base=os.path.basename(fn)
    return (info.is_dir() or not base or base.startswith("._") or
            fn.startswith("__MACOSX/") or "/." in fn or base.startswith("."))

def zip_to_images(zip_bytes: bytes) -> dict:
    out={}
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        for info in z.infolist():
            if _skip(info): continue
            base=os.path.basename(info.filename)
            if os.path.splitext(base)[1].lower() not in _IMG_EXTS: continue
            try:
                data=z.read(info); img=Image.open(io.BytesIO(data)).convert("RGB")
                out[_norm_key(base)]=img
            except UnidentifiedImageError:
                pass
    return out

def zip_to_masks(zip_bytes: bytes, thr=128) -> dict:
    out={}
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        for info in z.infolist():
            if _skip(info): continue
            base=os.path.basename(info.filename)
            if os.path.splitext(base)[1].lower() not in _MASK_EXTS: continue
            try:
                data=z.read(info); m=Image.open(io.BytesIO(data)).convert("L")
                out[_norm_key(base)] = (np.array(m)>=thr).astype(np.uint8)*255
            except UnidentifiedImageError:
                pass
    return out

In [ ]:
# Remediation

def suggest_remediation(dx, rv, pr, profile, anchors, cxcy, bgr_shape, target_bands, anchors_ok):
    """
    Generate specific crop/recompose recommendations based on metric deltas.

    Returns:
        dict with 'suggestions' (list of strings) and 'crop_box' (optional tuple)
    """
    H, W = bgr_shape[:2]
    cx, cy = cxcy
    suggestions = []
    crop_box = None  # (x1, y1, x2, y2) in pixel coordinates

    # Get target band centers
    dx_target = (target_bands["dx"][0] + target_bands["dx"][1]) / 2
    rv_target = (target_bands["rv"][0] + target_bands["rv"][1]) / 2
    pr_target = (target_bands["pr"][0] + target_bands["pr"][1]) / 2

    # === Δx Remediation ===
    if not in_band(dx, target_bands["dx"]):
        if profile == "GD-Edge" and anchors and anchors.get("vline"):
            x1, y1, x2, y2 = anchors["vline"]
            seam_x = 0.5 * (x1 + x2)
            current_dist = abs(cx - seam_x)
            target_dist = dx_target * W

            if dx < target_bands["dx"][0]:
                # Subject too close to seam - suggest moving subject or reframing
                suggestions.append(f"⚠️ Δx too low ({dx:.3f}): Subject is too close to the edge seam.")
                suggestions.append(f"   → Move subject {int((target_dist - current_dist))}px away from seam (currently {int(current_dist)}px)")
                suggestions.append(f"   → OR crop to reposition seam further from subject")
            elif dx > target_bands["dx"][1]:
                # Subject too far from seam
                suggestions.append(f"⚠️ Δx too high ({dx:.3f}): Subject is too far from edge seam.")
                suggestions.append(f"   → Move subject {int((current_dist - target_dist))}px closer to seam")
        else:
            # Center-based Δx (Horizon/Field or Edge without seam)
            current_offset = abs(cx - W/2)
            target_offset = dx_target * W

            if dx < target_bands["dx"][0]:
                suggestions.append(f"⚠️ Δx too low ({dx:.3f}): Subject too centered.")
                suggestions.append(f"   → Shift subject {int(target_offset - current_offset)}px horizontally from center")

                # Suggest asymmetric crop
                if cx < W/2:
                    crop_left = int(max(0, (W/2 - cx) - target_offset))
                    crop_box = (crop_left, 0, W, H)
                    suggestions.append(f"   → OR crop {crop_left}px from left edge to reframe")
                else:
                    crop_right = int(min(W, (cx - W/2) + target_offset))
                    crop_box = (0, 0, crop_right, H)
                    suggestions.append(f"   → OR crop to {crop_right}px width from left")

            elif dx > target_bands["dx"][1]:
                suggestions.append(f"⚠️ Δx too high ({dx:.3f}): Subject too far off-center.")
                suggestions.append(f"   → Move subject {int(current_offset - target_offset)}px toward center")

    # === r_v Remediation ===
    if not in_band(rv, target_bands["rv"]):
        if rv < target_bands["rv"][0]:
            deficit = target_bands["rv"][0] - rv
            suggestions.append(f"⚠️ r_v too low ({rv:.3f}): Not enough void/background space.")
            suggestions.append(f"   → Subject occupies {(1-rv)*100:.1f}% of frame, target is {(1-target_bands['rv'][0])*100:.1f}%")
            suggestions.append(f"   → Zoom out or use wider framing to add {deficit*100:.1f}% more negative space")

            # Suggest expanding canvas
            current_area_fraction = 1 - rv
            target_area_fraction = 1 - rv_target
            scale_factor = math.sqrt(current_area_fraction / target_area_fraction)
            new_W = int(W * scale_factor)
            new_H = int(H * scale_factor)
            suggestions.append(f"   → OR expand canvas to ~{new_W}×{new_H}px (keeping subject size)")

        elif rv > target_bands["rv"][1]:
            suggestions.append(f"⚠️ r_v too high ({rv:.3f}): Too much void space, subject too small.")
            suggestions.append(f"   → Subject only occupies {(1-rv)*100:.1f}% of frame, target is {(1-target_bands['rv'][1])*100:.1f}%")
            suggestions.append(f"   → Zoom in or crop tighter around subject")

            # Suggest crop to increase subject size
            current_area_fraction = 1 - rv
            target_area_fraction = 1 - rv_target
            scale_factor = math.sqrt(current_area_fraction / target_area_fraction)
            suggestions.append(f"   → Crop to ~{scale_factor:.1f}× tighter framing")

    # === σ_r (pr) Remediation ===
    if not in_band(pr, target_bands["pr"]):
        if pr < target_bands["pr"][0]:
            suggestions.append(f"⚠️ σ_r too low ({pr:.3f}): Surfaces too flat/quiet.")
            suggestions.append(f"   → Add texture, detail, or environmental complexity")
            suggestions.append(f"   → Check if background is too smooth/gradient")
            suggestions.append(f"   → Target surface complexity: {pr_target:.3f} (need +{(target_bands['pr'][0]-pr)*100:.1f}%)")

        elif pr > target_bands["pr"][1]:
            suggestions.append(f"⚠️ σ_r too high ({pr:.3f}): Surfaces too busy/noisy.")
            suggestions.append(f"   → Simplify background or reduce texture density")
            suggestions.append(f"   → Consider smoother surfaces or cleaner negative space")
            suggestions.append(f"   → Target surface complexity: {pr_target:.3f} (reduce by {(pr-target_bands['pr'][1])*100:.1f}%)")

    # === Anchor-specific ===
    if profile == "GD-Edge" and not anchors_ok:
        suggestions.append(f"❌ No edge seam detected for GD-Edge profile!")
        suggestions.append(f"   → Add or enhance a vertical edge/wall in the composition")
        suggestions.append(f"   → Ensure edge contrast is sufficient (try CLAHE option)")
        suggestions.append(f"   → OR switch to GD-Horizon or Field-Void profile")

    # === Summary action ===
    if not suggestions:
        suggestions.append("✅ All metrics within target ranges!")
    else:
        # Prioritize by severity
        priority_order = []
        if not anchors_ok and profile == "GD-Edge":
            priority_order.append("1st: Fix missing edge anchor")
        if rv < target_bands["rv"][0] or rv > target_bands["rv"][1]:
            priority_order.append(f"{'1st' if not priority_order else '2nd'}: Adjust void ratio (r_v)")
        if dx < target_bands["dx"][0] or dx > target_bands["dx"][1]:
            priority_order.append(f"{'1st' if not priority_order else str(len(priority_order)+1)+'th'}: Adjust offset (Δx)")
        if pr < target_bands["pr"][0] or pr > target_bands["pr"][1]:
            priority_order.append(f"{'1st' if not priority_order else str(len(priority_order)+1)+'th'}: Adjust surface roughness (σ_r)")

        if priority_order:
            suggestions.insert(0, "📋 Recommended priority: " + " → ".join(priority_order))

    return {"suggestions": suggestions, "crop_box": crop_box}


def visualize_remediation_crop(bgr, crop_box, output_path=None):
    """Draw the suggested crop box on the image."""
    if crop_box is None:
        return bgr

    vis = bgr.copy()
    x1, y1, x2, y2 = crop_box

    # Draw crop rectangle
    cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 255), 3)

    # Add corner markers
    corner_size = 20
    for (px, py) in [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]:
        cv2.line(vis, (px-corner_size, py), (px+corner_size, py), (0, 255, 0), 2)
        cv2.line(vis, (px, py-corner_size), (px, py+corner_size), (0, 255, 0), 2)

    # Add label
    cv2.putText(vis, "Suggested Crop", (x1+10, y1+30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2, cv2.LINE_AA)

    return vis


print("Remediation functions loaded. Use suggest_remediation() in your scoring workflow.")

In [ ]:
# Single-image scorer UI
import ipywidgets as W
from IPython.display import display, HTML
from google.colab.patches import cv2_imshow

img_u  = W.FileUpload(accept='.png,.jpg,.jpeg,.webp', multiple=False, description='Image')
mask_u = W.FileUpload(accept='.png,.jpg,.jpeg', multiple=False, description='Mask (opt)')
mode_dd= W.Dropdown(options=['Auto','GD-Edge','GD-Horizon','Field-Void'], value='Auto', description='Profile')
clahe_cb = W.Checkbox(value=False, description='CLAHE anchors')
span_dd  = W.Dropdown(options=[('Relaxed (0.45)',0.45), ('Very (0.40)',0.40), ('Strict (0.60)',0.60)],
                      value=0.45, description='Span')
run_btn = W.Button(description='Score image', button_style='success')
sout = W.Output()

display(W.HBox([img_u, mask_u]))
display(W.HBox([mode_dd, clahe_cb, span_dd, run_btn]))
display(sout)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

def _bytes_to_bgr(b):
    pil = Image.open(io.BytesIO(b)).convert('RGB')
    return cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)

def on_single(_):
    with sout:
        sout.clear_output()
        if not img_u.value:
            print("Upload an image."); return
        ikey = list(img_u.value.keys())[0]
        bgr  = _bytes_to_bgr(img_u.value[ikey]['content'])
        mask = None
        if mask_u.value:
            mkey = list(mask_u.value.keys())[0]
            mm   = Image.open(io.BytesIO(mask_u.value[mkey]['content'])).convert('L')
            mask = (np.array(mm)>127).astype(np.uint8)*255

        if mode_dd.value == 'Auto':
            prof, info = guess_profile(bgr, mask, use_clahe=clahe_cb.value, span_frac=span_dd.value)
            is_ambiguous = info.get("is_ambiguous", False)

            if is_ambiguous:
                probs = info.get("probabilities", {})
                print(f"⚠️  AMBIGUOUS PROFILE (no clear winner):")
                for p, score in sorted(probs.items(), key=lambda x: -x[1]):
                    print(f"   {p}: {score:.2%}")
                print(f"   → Using {prof} but consider manual override\n")
        else:
            prof = mode_dd.value
            info = {"edge_conf":0.0, "horiz_conf":0.0, "field_conf":0.0}
            is_ambiguous = False

        anchors = {}
        if prof == 'GD-Edge':
            vlines = detect_lines(bgr, vertical=True, use_clahe=clahe_cb.value, span_frac=span_dd.value)
            m2 = ensure_mask2d(mask, bgr)

            # Filter to background-only lines
            bg_vlines = []
            H, W = bgr.shape[:2]
            min_length = max(H, W) * 0.15

            for (x1, y1, x2, y2) in vlines:
                line_length = np.sqrt((x2-x1)**2 + (y2-y1)**2)
                if line_length < min_length:
                    continue

                num_samples = 10
                bg_count = 0  # Fixed: was "bg count"
                for i in range(num_samples):
                    t = i / float(num_samples - 1)
                    px = int(x1 + t * (x2 - x1))
                    py = int(y1 + t * (y2 - y1))
                    if 0 <= py < m2.shape[0] and 0 <= px < m2.shape[1]:
                        if m2[py, px] == 0:
                            bg_count += 1

                if bg_count / float(num_samples) > 0.7:
                    bg_vlines.append((x1, y1, x2, y2))

            xs = np.nonzero(m2)[1]
            cx = (xs.mean() if xs.size else bgr.shape[1]/2.0)
            best = pick_nearest_vertical(bg_vlines, float(cx)) if bg_vlines else None
            if best is not None:
                anchors['vline'] = best

        dx, rv, pr, cc = compute_metrics(bgr, mask, prof, anchors, use_clahe=clahe_cb.value)
        anchors_ok = True if (prof != 'GD-Edge') else ('vline' in anchors)
        artist_pass, engine_pass, dx_out, rv_out, pr_out = score_image(dx, rv, pr, prof, anchors_ok)
        state, metrics_in, closest = classify_state(dx, rv, pr, prof, anchors_ok,
                                                    artist_pass, engine_pass,
                                                    dx_out, rv_out, pr_out)

        # Calculate reason_for_fail and deltas
        if not engine_pass:
            reasons = []
            if not anchors_ok and prof == 'GD-Edge':
                reasons.append("no_anchor")
            if dx_out[0] > 0:
                reasons.append("dx_too_low")
            if dx_out[1] > 0:
                reasons.append("dx_too_high")
            if rv_out[0] > 0:
                reasons.append("rv_too_low")
            if rv_out[1] > 0:
                reasons.append("rv_too_high")
            if pr_out[0] > 0:
                reasons.append("pr_too_low")
            if pr_out[1] > 0:
                reasons.append("pr_too_high")
            reason_for_fail = "; ".join(reasons) if reasons else "unknown"
        else:
            reason_for_fail = ""

        # Get the target bands for this profile
        if prof == "GD-Edge":
            target_bands = ENG_EDGE
        elif prof == "GD-Horizon":
            target_bands = {"dx":(0.12,0.22), "rv":HORIZON_RV, "pr":(0.18,0.32)}
        else:
            target_bands = {"dx":(0.00,1.00), "rv":(0.74,0.88), "pr":(0.20,0.34)}

        delta_dx = dx - (target_bands["dx"][0] + target_bands["dx"][1]) / 2
        delta_rv = rv - (target_bands["rv"][0] + target_bands["rv"][1]) / 2

        df = pd.DataFrame([{
            "profile": prof, "anchors_ok": anchors_ok, "is_ambiguous": is_ambiguous,
            "dx": round(dx,4), "rv": round(rv,4), "pr": round(pr,4),
            "state": state, "metrics_in_range": f"{metrics_in}/3", "closest_metric": closest,
            "artist_pass": artist_pass, "deploy_pass": engine_pass,
            "reason_for_fail": reason_for_fail,
            "delta_dx": round(delta_dx,4),
            "delta_rv": round(delta_rv,4),
            "dx_below": round(dx_out[0],3), "dx_above": round(dx_out[1],3),
            "rv_below": round(rv_out[0],3), "rv_above": round(rv_out[1],3),
            "pr_below": round(pr_out[0],3), "pr_above": round(pr_out[1],3),
            "edge_conf": info.get("edge_conf",0.0),
            "horiz_conf": info.get("horiz_conf",0.0),
            "field_conf": info.get("field_conf",0.0),
        }])
        display(df)
        display(HTML("<small><b>Anchors OK</b> = seam detected when required (GD–Edge only). "
                     "<b>Deploy</b> = engine-bias window.</small>"))

        # === REMEDIATION SUGGESTIONS ===
        remediation = suggest_remediation(dx, rv, pr, prof, anchors, cc, bgr.shape,
                                         target_bands, anchors_ok)

        print("\n" + "="*80)
        print("🔧 REMEDIATION SUGGESTIONS")
        print("="*80)
        for suggestion in remediation["suggestions"]:
            print(suggestion)
        print("="*80 + "\n")

        info["state"] = state
        ov = draw_overlay(bgr, mask, prof, anchors, dx, rv, pr, cc, info)
        cv2_imshow(ov[:,:,::-1])

run_btn.on_click(on_single)

In [ ]:
# Batch scorer UI
import ipywidgets as W
from IPython.display import display, HTML
from datetime import datetime
from pathlib import Path

imgs_zip = W.FileUpload(accept='.zip', multiple=False, description='Images ZIP')
masks_zip= W.FileUpload(accept='.zip', multiple=False, description='Masks ZIP (opt)')
clahe_b  = W.Checkbox(value=False, description='CLAHE anchors')
span_b   = W.Dropdown(options=[('Relaxed (0.45)',0.45), ('Very (0.40)',0.40), ('Strict (0.60)',0.60)],
                      value=0.45, description='Span')
run_b    = W.Button(description='Run batch', button_style='info')
bout     = W.Output()

display(W.HBox([imgs_zip, masks_zip]))
display(W.HBox([clahe_b, span_b, run_b]))
display(bout)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

def on_batch(_):
    with bout:
        bout.clear_output()
        if not imgs_zip.value:
            print("Upload an images ZIP."); return
        ik = list(imgs_zip.value.keys())[0]
        imgs = zip_to_images(imgs_zip.value[ik]['content'])
        masks={}
        if masks_zip.value:
            mk = list(masks_zip.value.keys())[0]
            masks = zip_to_masks(masks_zip.value[mk]['content'])
        print(f"parsed: {len(imgs)} images, {len(masks)} masks")

        rows=[]
        for name, pil in imgs.items():
            bgr  = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
            mask = masks.get(name, None)
            prof, info = guess_profile(bgr, mask, use_clahe=clahe_b.value, span_frac=span_b.value)
            # Track ambiguous cases for batch report
            is_ambiguous = info.get("is_ambiguous", False)

            anchors = {}
            if prof == 'GD-Edge':
                vlines = detect_lines(bgr, vertical=True, use_clahe=clahe_b.value, span_frac=span_b.value)
                m2 = ensure_mask2d(mask, bgr)

                # Filter to background-only lines
                bg_vlines = []
                H, W = bgr.shape[:2]
                min_length = max(H, W) * 0.15

                for (x1, y1, x2, y2) in vlines:
                    line_length = np.sqrt((x2-x1)**2 + (y2-y1)**2)
                    if line_length < min_length:
                        continue

                    num_samples = 10
                    bg_count = 0  # Fixed: was "bg count"
                    for i in range(num_samples):
                        t = i / float(num_samples - 1)
                        px = int(x1 + t * (x2 - x1))
                        py = int(y1 + t * (y2 - y1))
                        if 0 <= py < m2.shape[0] and 0 <= px < m2.shape[1]:
                            if m2[py, px] == 0:
                                bg_count += 1

                    if bg_count / float(num_samples) > 0.7:
                        bg_vlines.append((x1, y1, x2, y2))

                xs = np.nonzero(m2)[1]
                cx = (xs.mean() if xs.size else bgr.shape[1]/2.0)
                best = pick_nearest_vertical(bg_vlines, float(cx)) if bg_vlines else None
                if best is not None:
                    anchors['vline'] = best

            dx, rv, pr, cc = compute_metrics(bgr, mask, prof, anchors, use_clahe=clahe_b.value)
            anchors_ok = True if (prof != 'GD-Edge') else ('vline' in anchors)
            artist_pass, engine_pass, dx_out, rv_out, pr_out = score_image(dx, rv, pr, prof, anchors_ok)
            state, metrics_in, closest = classify_state(dx, rv, pr, prof, anchors_ok,
                                                        artist_pass, engine_pass,
                                                        dx_out, rv_out, pr_out)
            area, conv = mask_quality(ensure_mask2d(mask, bgr))

             # Calculate reason_for_fail and deltas
            if not engine_pass:
                reasons = []
                if not anchors_ok and prof == 'GD-Edge':
                    reasons.append("no_anchor")
                if dx_out[0] > 0:
                    reasons.append("dx_too_low")
                if dx_out[1] > 0:
                    reasons.append("dx_too_high")
                if rv_out[0] > 0:
                    reasons.append("rv_too_low")
                if rv_out[1] > 0:
                    reasons.append("rv_too_high")
                if pr_out[0] > 0:
                    reasons.append("pr_too_low")
                if pr_out[1] > 0:
                    reasons.append("pr_too_high")
                reason_for_fail = "; ".join(reasons) if reasons else "unknown"
            else:
                reason_for_fail = ""

            # Get the target bands for this profile
            if prof == "GD-Edge":
                target_bands = ENG_EDGE
            elif prof == "GD-Horizon":
                target_bands = {"dx":(0.12,0.22), "rv":HORIZON_RV, "pr":(0.18,0.32)}
            else:  # Field-Void
                target_bands = {"dx":(0.00,1.00), "rv":(0.74,0.88), "pr":(0.20,0.34)}

            #Calculate deltas to center of target band
            delta_dx = dx - (target_bands["dx"][0] + target_bands["dx"][1]) / 2
            delta_rv = rv - (target_bands["rv"][0] + target_bands["rv"][1]) / 2

            rows.append({
                "name": name, "profile": prof, "anchors_ok": anchors_ok, "is_ambiguous": is_ambiguous,
                "dx": round(dx,4), "rv": round(rv,4), "pr": round(pr,4),
                "state": state, "metrics_in_range": f"{metrics_in}/3", "closest_metric": closest,
                "artist_pass": artist_pass, "deploy_pass": engine_pass,
                "reason_for_fail": reason_for_fail, "delta_dx": round(delta_dx,4), "delta_rv": round(delta_rv,4),
                "dx_below": round(dx_out[0],3), "dx_above": round(dx_out[1],3),
                "rv_below": round(rv_out[0],3), "rv_above": round(rv_out[1],3),
                "pr_below": round(pr_out[0],3), "pr_above": round(pr_out[1],3),
                "mask_area": round(area,3), "mask_convex": round(conv,3),
                "edge_conf": info.get("edge_conf",0.0),
                "horiz_conf": info.get("horiz_conf",0.0),
                "field_conf": info.get("field_conf",0.0),
            })

        df = pd.DataFrame(rows); display(df.head(12))
        display(HTML("<small><b>Anchors OK</b> = seam detected when required (GD-Edge). "
                     "<b>Deploy</b> = engine window.</small>"))
        out_csv = Path('/content')/f"ocf_batch_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        out_csv.parent.mkdir(parents=True, exist_ok=True); df.to_csv(out_csv, index=False)
        print(f"Saved CSV → {out_csv}")

run_b.on_click(on_batch)

In [ ]:
import inspect
print("PR_MODE active:", PR_MODE)
print("compute_metrics id:", id(compute_metrics))
print("compute_metrics lines:", len(inspect.getsource(compute_metrics).splitlines()))